# Week 11: Clustering & Boundary Tightening

## Strategy: Cluster-Centric Trust Regions
This week, we analyze the search space through a **clustering lens**. Instead of defining a rigid Trust Region around a single best point, we:
1. **Identify the High-Yield Cluster:** Select the top $K$ performing queries.
2. **Calculate the Centroid:** Use the mean of this cluster as the starting anchor for optimization.
3. **Boundary Tightening:** Use the standard deviation (spread) of this cluster to dynamically size the Trust Region. Dense clusters result in tighter, exploitation-heavy boundaries.
4. **Noise Filtering:** Ignore the low-performing scattered points (treating them as noise).

In [1]:
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import sys
import os

# Ensure we can import from src
sys.path.append(os.path.abspath('..'))
from src.utils import load_data

warnings.filterwarnings("ignore")
np.random.seed(51) # Week 11 Seed

print("Ready for Cluster-Based Optimization")

Ready for Cluster-Based Optimization


In [2]:
def suggest_next_point_clustered(func_id, X_train, y_train):
    print(f"\n--- Optimizing Function {func_id} (Clustered Bounds) ---")
    
    # 1. Preprocessing
    scaler_x = StandardScaler()
    X_scaled = scaler_x.fit_transform(X_train)
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    
    # 2. Train Surrogate Models (Scaled Ensemble N=20)
    hidden_layers = (128, 64) if func_id in [7, 8] else (64, 32)
    base_mlp = MLPRegressor(hidden_layer_sizes=hidden_layers, alpha=0.01, 
                            activation='tanh', solver='lbfgs', max_iter=2000)
    
    regr = BaggingRegressor(estimator=base_mlp, n_estimators=20, random_state=42, n_jobs=-1)
    regr.fit(X_scaled, y_scaled)
    
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)
    gp_model = GaussianProcessRegressor(kernel=kernel, normalize_y=False)
    gp_model.fit(X_scaled, y_scaled)
    
    # 3. CLUSTERING LOGIC: Find the "High-Yield Cluster"
    k_cluster = max(3, len(y_train) // 5) # Take top 15-20% of points
    top_indices = np.argsort(y_train)[-k_cluster:]
    top_X_real = X_train[top_indices]
    
    # Centroid Trend
    centroid_X_real = np.mean(top_X_real, axis=0)
    # Boundary Tightening (based on cluster standard deviation)
    cluster_std_real = np.std(top_X_real, axis=0)
    
    # Dynamic radius: Tighten if points are clustered, expand if scattered
    # We clip to [0.05, 0.25] to prevent the box from becoming exactly zero or too large
    dynamic_radius = np.clip(cluster_std_real * 2.0, 0.05, 0.25)
    
    # Print Clustering Diagnostics
    print(f"   [Cluster Info] Top {k_cluster} points forming the cluster.")
    avg_radius = np.mean(dynamic_radius)
    print(f"   [Cluster Info] Average Dynamic Radius (Boundary Tightening): {avg_radius:.4f}")

    # Calculate bounds in scaled space for the optimizer
    bounds_scaled = []
    for i in range(X_train.shape[1]):
        mean_val, scale_val = scaler_x.mean_[i], scaler_x.scale_[i]
        c_val = centroid_X_real[i]
        r_val = dynamic_radius[i]
        
        lower = (max(0.0, c_val - r_val) - mean_val) / scale_val
        upper = (min(1.0, c_val + r_val) - mean_val) / scale_val
        bounds_scaled.append((lower, upper))
        
    centroid_X_scaled = scaler_x.transform(centroid_X_real.reshape(1, -1)).flatten()
    
    # 4. Objective Function (UCB)
    def objective_function(x):
        x_reshaped = x.reshape(1, -1)
        nn_preds = np.array([est.predict(x_reshaped)[0] for est in regr.estimators_])
        avg_nn, std_nn = np.mean(nn_preds), np.std(nn_preds)
        
        gp_pred, gp_std = gp_model.predict(x_reshaped, return_std=True)
        gp_pred, gp_std = gp_pred[0], gp_std[0]
        
        comb_mean = 0.6 * avg_nn + 0.4 * gp_pred
        comb_std = 0.6 * std_nn + 0.4 * gp_std
        
        ucb = comb_mean + 1.96 * comb_std
        
        # Repulsion from all points in the top cluster to explore *within* the cluster
        penalty = 0
        top_X_scaled = scaler_x.transform(top_X_real)
        for p_scaled in top_X_scaled:
            dist_sq = np.sum((x_reshaped - p_scaled.reshape(1, -1))**2)
            penalty += 5.0 * np.exp(-dist_sq / (2 * 0.1**2))
            
        return -ucb + penalty

    # 5. Optimize within Clustered Bounds
    x_init = centroid_X_scaled + np.random.uniform(-0.05, 0.05, size=centroid_X_scaled.shape)
    res = minimize(fun=objective_function, x0=x_init, method='L-BFGS-B', 
                   bounds=bounds_scaled, options={'maxiter': 100})
    
    next_point = np.clip(scaler_x.inverse_transform(res.x.reshape(1, -1)).flatten(), 0.0, 1.0)
    return next_point

In [3]:
submission_queries = {}
print("Starting Round 11 Evaluation (Clustering)...")

for func_id in range(1, 9):
    # Ensure you have updated data to 20 points before running
    X_known, y_known = load_data(func_id)
    next_x = suggest_next_point_clustered(func_id, X_known, y_known)
    submission_queries[func_id] = next_x

print("\n" + "="*30)
print("FORMATTED SUBMISSION OUTPUT")
print("="*30)

for func_id, x_val in submission_queries.items():
    formatted_str = "-".join([f"{val:.6f}" for val in x_val])
    print(f"function_number: {func_id}: {formatted_str}")

   [Cluster Info] Top 10 points forming the cluster.
   [Cluster Info] Average Dynamic Radius (Boundary Tightening): 0.2028

FORMATTED SUBMISSION OUTPUT
function_number: 1: 0.486071-0.485050
function_number: 2: 0.557918-1.000000
function_number: 3: 0.578573-0.352227-0.564527
function_number: 4: 0.489785-0.464926-0.386536-0.430105
function_number: 5: 0.919412-0.917274-0.949130-1.000000
function_number: 6: 0.380729-0.452360-0.541026-0.996969-0.000000
function_number: 7: 0.000000-0.189799-0.208667-0.348754-0.347568-0.796759
function_number: 8: 0.000000-0.000000-0.000000-0.000000-0.781727-0.493721-0.372291-0.601565
